# AlphaFold 3 inference with pre-computed data pipeline output

*Still experimental/work-in-progress*

This implements
[data pipeline re-use](https://github.com/google-deepmind/alphafold3/blob/main/docs/performance.md#pre-computing-and-reusing-msa-and-templates) with the following addons:
1. Data pipeline output is only ever stored as (gzip-compressed) JSON files.
2. Input sequences are matched to pre-computed data pipeline output **by sequence**. There's no need to track sequences by using a particular set of identifiers (Uniprot/Ensembl/RefSeq) as file names.
3. The pipeline automatically identifies input sequences that do not have pre-computed data pipeline output. It will then run the data pipeline only on the missing sequences, with each missing sequence as a separate SLURM job.

In [2]:
# Adjust settings in `config.yaml`:
# - Use human pre-computed MSAs by specifying data-sources
# - Reduced number of recycles for faster inference (at reduced accuracy)
cat config.yaml

alphafold3:
  data_sources: >-
    --data_dir=/cluster/project/beltrao/shared/25.05_alphafold3_msas_human
  predictions:
    run_alphafold_args: >-
      --num_recycles=3


In [3]:
# Input sequences:
# Ubiquitin (from Wikipedia): MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG
# Two ubiquitin-binding proteins (with pre-computed MSAs): USP12/O75317 and UBE2V1/Q13404
#af3io input-show /cluster/project/beltrao/shared/25.05_alphafold3_msas_human/o75317_data.json.gz
af3io input-show /cluster/project/beltrao/shared/25.05_alphafold3_msas_human/q13404_data.json.gz

{
  "dialect": "alphafold3",
  "version": 2,
  "name": "q13404",
  "sequences": [
    {
      "protein": {
        "id": "A",
        "sequence": "MAATTGSGVKVPRNFRLLEELEEGQKGVGDGTVSWGLEDDEDMTLTRWTGMIIGPPRTIYENRIYSLKIECGPKYPEAPPFVRFVTKINMNGVNSSNGVVDPRAISVLAKWQNSYSIKVVLQELRRLMMSKENMKLPQPPEGQCYSN",
        "modifications": [],
        "unpairedMsa": "<5,878 sequences, 1.58 MB, hash: 6d97d6>",
        "pairedMsa": "<10,950 sequences, 3.2 MB, hash: e4b68b>",
        "templates": "<4 templates, 595.57 KB, hash: fcb15c>"
      }
    }
  ],
  "modelSeeds": [
    4
  ],
  "bondedAtomPairs": null,
  "userCCD": null
}


In [4]:
# Create two input JSON files with a known ubiquitin-binding protein (o75317 and q13404), and ubiquitin
# o75317 and q13404 will have pre-computed MSAs whereas ubiquitin does not
af3io input-create alphafold3_jsons/o75317_ubiquitin.json \
    --sequence MEILMTVSKFASICTMGANASALEKEIGPEQFPVNEHYFGLVNFGNTCYCNSVLQALYFCRPFREKVLAYKSQPRKKESLLTCLADLFHSIATQKKKVGVIPPKKFITRLRKENELFDNYMQQDAHEFLNYLLNTIADILQEERKQEKQNGRLPNGNIDNENNNSTPDPTWVHEIFQGTLTNETRCLTCETISSKDEDFLDLSVDVEQNTSITHCLRGFSNTETLCSEYKYYCEECRSKQEAHKRMKVKKLPMILALHLKRFKYMDQLHRYTKLSYRVVFPLELRLFNTSGDATNPDRMYDLVAVVVHCGSGPNRGHYIAIVKSHDFWLLFDDDIVEKIDAQAIEEFYGLTSDISKNSESGYILFYQSRD \
    --sequence MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG
af3io input-create alphafold3_jsons/q13404_ubiquitin.json \
    --sequence MAATTGSGVKVPRNFRLLEELEEGQKGVGDGTVSWGLEDDEDMTLTRWTGMIIGPPRTIYENRIYSLKIECGPKYPEAPPFVRFVTKINMNGVNSSNGVVDPRAISVLAKWQNSYSIKVVLQELRRLMMSKENMKLPQPPEGQCYSN \
    --sequence MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG
echo Contents of alphafold3_jons/
ls -1 alphafold3_jsons/

Setting name to: o75317_ubiquitin
Write:	/cluster/project/beltrao/jjaenes/26.06_batch-infer/examples/alphafold3_datafill/alphafold3_jsons/o75317_ubiquitin.json
Setting name to: q13404_ubiquitin
Write:	/cluster/project/beltrao/jjaenes/26.06_batch-infer/examples/alphafold3_datafill/alphafold3_jsons/q13404_ubiquitin.json
Contents of alphafold3_jons/
o75317_ubiquitin.json
q13404_ubiquitin.json


In [5]:
# Negative control - two ubiquitin-interacting proteins (that would be unlikely to interact with each other)
af3io input-create alphafold3_jsons/o75317_q13404.json \
    --sequence MEILMTVSKFASICTMGANASALEKEIGPEQFPVNEHYFGLVNFGNTCYCNSVLQALYFCRPFREKVLAYKSQPRKKESLLTCLADLFHSIATQKKKVGVIPPKKFITRLRKENELFDNYMQQDAHEFLNYLLNTIADILQEERKQEKQNGRLPNGNIDNENNNSTPDPTWVHEIFQGTLTNETRCLTCETISSKDEDFLDLSVDVEQNTSITHCLRGFSNTETLCSEYKYYCEECRSKQEAHKRMKVKKLPMILALHLKRFKYMDQLHRYTKLSYRVVFPLELRLFNTSGDATNPDRMYDLVAVVVHCGSGPNRGHYIAIVKSHDFWLLFDDDIVEKIDAQAIEEFYGLTSDISKNSESGYILFYQSRD \
    --sequence MAATTGSGVKVPRNFRLLEELEEGQKGVGDGTVSWGLEDDEDMTLTRWTGMIIGPPRTIYENRIYSLKIECGPKYPEAPPFVRFVTKINMNGVNSSNGVVDPRAISVLAKWQNSYSIKVVLQELRRLMMSKENMKLPQPPEGQCYSN

Setting name to: o75317_q13404
Write:	/cluster/project/beltrao/jjaenes/26.06_batch-infer/examples/alphafold3_datafill/alphafold3_jsons/o75317_q13404.json


In [2]:
# `alphafold3_datafill_missing` checks all input sequences against `data_sources` to find sequences without pre-computed data pipeline output:
#  - reads all input JSONs from `alphafold3_jsons/`
#  - writes missing sequence input JSONs to `alphafold3_missing/`
batch-infer start alphafold3_datafill_missing

In [3]:
# For every missing sequence (ubiqutin), there's now an input JSON under alphafold3_missing/
# The file names consist of the original input JSON, and the id of the missing sequence
# e.g. o75317_ubiquitin_b refers to sequence B from alphafold3_jsons/ao75317_ubiquitin.json
ls -l alphafold3_missing/

total 4
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 331 Jul 28 10:55 o75317_ubiquitin_b.json


In [7]:
# `alphafold3_datafill_msas` runs AlphaFold 3 data pipeline for missing sequences:
# - runs data pipeline for all missing sequence from `alphafold3_missing/`
# - writes the output to `alphafold3_msas/`
batch-infer start alphafold3_datafill_msas

In [9]:
# One job per every missing chain/file under alphafold3_missing/
batch-infer status

  JOBID  PARTITION    NAME                                                                    STATE       TIME  TIME_LIMIT  NODELIST(REASON)
8862647  normal.120h  batch_infer:alphafold3_datafill_msas                                    RUNNING     8:45  7-00:00:00  eu-a2p-495
8863231  normal.4h    batch-infer:8862647:alphafold3_datafill_msas_run:id=o75317_ubiquitin_b  RUNNING     5:33     4:00:00  eu-a2p-304


In [11]:
# Data pipeline output for missing sequence(s) stored under alphafold3_msas/
ls -l alphafold3_msas/

total 2088
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 2121804 Jul 28 13:19 o75317_ubiquitin_b_data.json.gz


In [10]:
# `alphafold3_datafill_predictions` runs the inference step:
# - reads input JSONs from `alphafold3_jsons/`
# - creates data pipeline output (in local scratch) based `data_sources` and/or `alphafold3_msas/`
# - writes structure predictions to `alphafold3_predictions/`
batch-infer start alphafold3_datafill_predictions

In [14]:
# One job per prediction
batch-infer status

  JOBID  PARTITION    NAME                                                           STATE       TIME  TIME_LIMIT  NODELIST(REASON)
8870774  gpupr.4h     batch-infer:8870504:alphafold3_datafill_predictions_batch1_1:  RUNNING     0:45     4:00:00  eu-a65-04
8870773  gpupr.4h     batch-infer:8870504:alphafold3_datafill_predictions_batch0_1:  RUNNING     0:47     4:00:00  eu-a65-04
8870769  gpupr.4h     batch-infer:8870504:alphafold3_datafill_predictions_batch2_1:  RUNNING     0:49     4:00:00  eu-a65-04
8870504  normal.120h  batch_infer:alphafold3_datafill_predictions                    RUNNING     1:02  7-00:00:00  eu-a2p-514


In [15]:
# Predictions stored as zip-compressed archives under alphafold3_predictions/
ls -l alphafold3_predictions/

total 4924
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 2523866 Jul 28 13:46 o75317_q13404.zip
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 1855478 Jul 28 13:46 o75317_ubiquitin.zip
-rw-r--r-- 1 jjaenes biol-imsb-beltrao  634288 Jul 28 13:46 q13404_ubiquitin.zip


In [16]:
# Positive controls have high ipTM (>0.5)
unzip -p alphafold3_predictions/o75317_ubiquitin.zip o75317_ubiquitin/o75317_ubiquitin_summary_confidences.json | jello '_["chain_pair_iptm"][0][1]'
unzip -p alphafold3_predictions/q13404_ubiquitin.zip q13404_ubiquitin/q13404_ubiquitin_summary_confidences.json | jello '_["chain_pair_iptm"][0][1]'

0.95
0.79


In [17]:
# Negative control has low ipTM (<0.5)
unzip -p alphafold3_predictions/o75317_q13404.zip o75317_q13404/o75317_q13404_summary_confidences.json | jello '_["chain_pair_iptm"][0][1]'

0.13
